# OmniVoice - Deploy trên Google Colab

Notebook này cài môi trường, tải model `k2-fsa/OmniVoice`, dựng giao diện Gradio và mở public URL bằng `share=True`.

## Cách dùng nhanh
1. Vào `Runtime` → `Change runtime type` → chọn `T4 GPU` hoặc GPU tốt hơn.
2. Chạy lần lượt các cell từ trên xuống.
3. Mở link Gradio public được in ra ở cell cuối.

> Gợi ý: nếu chỉ clone giọng và bạn tự nhập reference text, có thể để `LOAD_ASR=False` để giảm RAM/VRAM và khởi động nhanh hơn.


In [ ]:
#@title 1) Kiểm tra GPU
import os, sys, subprocess, textwrap, json, platform

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA:', torch.version.cuda)
except Exception as e:
    print('Torch chưa sẵn sàng:', repr(e))

!nvidia-smi || true


In [ ]:
#@title 2) Clone project hoặc dùng thư mục hiện có
# Nếu mở notebook trực tiếp từ repo đã clone trong Colab thì để USE_EXISTING=True.
USE_EXISTING = False  #@param {type:"boolean"}
REPO_URL = 'https://github.com/k2-fsa/OmniVoice-Space.git'  #@param {type:"string"}
PROJECT_DIR = '/content/OmniVoice-Space'  #@param {type:"string"}

import os, subprocess, pathlib

if USE_EXISTING and os.path.exists('/content/drive/MyDrive/OmniVoice-Space'):
    PROJECT_DIR = '/content/drive/MyDrive/OmniVoice-Space'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning {REPO_URL} -> {PROJECT_DIR}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, PROJECT_DIR])
else:
    print(f'Project directory exists: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
print('Files:', os.listdir('.')[:20])


In [ ]:
#@title 3) Cài system packages và Python dependencies
import os, subprocess, sys

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!apt-get update -qq
!apt-get install -y -qq ffmpeg git-lfs

# Colab thường đã có torch CUDA. Cài theo requirements trước; nếu torch bị conflict, restart runtime rồi chạy lại.
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt

# Một số dependency có thể chưa nằm trong requirements.txt nhưng project training/data có import.
!python -m pip install -U webdataset huggingface_hub soundfile pydub accelerate

# Cho phép import package local
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [ ]:
#@title 4) Cấu hình deploy
MODEL_ID = 'k2-fsa/OmniVoice'  #@param {type:"string"}
LOAD_ASR = False  #@param {type:"boolean"}
DEFAULT_NUM_STEP = 16  #@param {type:"slider", min:8, max:32, step:4}
DEFAULT_GUIDANCE_SCALE = 2.0  #@param {type:"number"}
SHARE_PUBLIC_LINK = True  #@param {type:"boolean"}

print('MODEL_ID:', MODEL_ID)
print('LOAD_ASR:', LOAD_ASR)
print('DEFAULT_NUM_STEP:', DEFAULT_NUM_STEP)
print('DEFAULT_GUIDANCE_SCALE:', DEFAULT_GUIDANCE_SCALE)
print('SHARE_PUBLIC_LINK:', SHARE_PUBLIC_LINK)


In [ ]:
#@title 5) Load model
import gc, os, sys, torch

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Loading model {MODEL_ID} on {device} with dtype={dtype} ...')
model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map=device,
    dtype=dtype,
    load_asr=LOAD_ASR,
)
model.eval()
sampling_rate = model.sampling_rate
print('Loaded. Sampling rate:', sampling_rate)


In [ ]:
#@title 6) Tạo Gradio app tối ưu cho Colab
import hashlib, json, tempfile, os, gc
from typing import Any, Dict

import gradio as gr
import numpy as np
import torch

from omnivoice import OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

ALL_LANGUAGES = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)
prompt_cache = {}

def _file_hash(path):
    if not path:
        return 'none'
    h = hashlib.sha1()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def _get_voice_prompt(ref_audio, ref_text, preprocess_prompt):
    key = (_file_hash(ref_audio), ref_text or '', bool(preprocess_prompt))
    if key not in prompt_cache:
        prompt_cache[key] = model.create_voice_clone_prompt(
            ref_audio=ref_audio,
            ref_text=ref_text or None,
            preprocess_prompt=bool(preprocess_prompt),
        )
    return prompt_cache[key]

def clear_prompt_cache():
    prompt_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return 'Đã xóa cache prompt.'

def generate_audio(
    text, language, mode, ref_audio, ref_text, instruct,
    num_step, guidance_scale, denoise, speed, duration,
    preprocess_prompt, postprocess_output,
):
    if not text or not text.strip():
        return None, 'Vui lòng nhập text cần đọc.'

    gen_config = OmniVoiceGenerationConfig(
        num_step=int(num_step or DEFAULT_NUM_STEP),
        guidance_scale=float(guidance_scale if guidance_scale is not None else DEFAULT_GUIDANCE_SCALE),
        denoise=bool(denoise),
        preprocess_prompt=bool(preprocess_prompt),
        postprocess_output=bool(postprocess_output),
    )

    lang = language if language and language != 'Auto' else None
    kwargs = dict(text=text.strip(), language=lang, generation_config=gen_config)

    if speed is not None and float(speed) != 1.0:
        kwargs['speed'] = float(speed)
    if duration is not None and float(duration) > 0:
        kwargs['duration'] = float(duration)

    if mode == 'Clone voice':
        if not ref_audio:
            return None, 'Vui lòng upload reference audio khi dùng Clone voice.'
        kwargs['voice_clone_prompt'] = _get_voice_prompt(ref_audio, ref_text, preprocess_prompt)
    elif mode == 'Voice design' and instruct and instruct.strip():
        kwargs['instruct'] = instruct.strip()

    try:
        with torch.inference_mode():
            audio = model.generate(**kwargs)[0]
        waveform = np.clip(audio, -1.0, 1.0)
        waveform_i16 = (waveform * 32767).astype(np.int16)
        return (sampling_rate, waveform_i16), 'Xong.'
    except Exception as e:
        return None, f'Lỗi: {type(e).__name__}: {e}'

css = '''
.gradio-container {max-width: 1100px !important;}
.compact-audio audio {height: 64px !important;}
'''

with gr.Blocks(title='OmniVoice Colab', theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown('# OmniVoice Colab Demo')
    gr.Markdown('Voice cloning / voice design / auto voice. Public link chạy qua Gradio share.')

    with gr.Row():
        with gr.Column(scale=2):
            text = gr.Textbox(
                label='Text cần tổng hợp',
                lines=5,
                value='Hello, this is OmniVoice running on Google Colab.',
            )
            language = gr.Dropdown(choices=ALL_LANGUAGES, value='Auto', label='Language')
            mode = gr.Radio(
                choices=['Auto voice', 'Clone voice', 'Voice design'],
                value='Clone voice',
                label='Mode',
            )
            ref_audio = gr.Audio(label='Reference audio', type='filepath')
            ref_text = gr.Textbox(label='Reference text (khuyến nghị nhập để khỏi dùng ASR)', lines=2)
            instruct = gr.Textbox(label='Voice design instruct', placeholder='male, British accent', lines=1)

        with gr.Column(scale=1):
            output = gr.Audio(label='Output', elem_classes=['compact-audio'])
            status = gr.Textbox(label='Status', interactive=False)
            run_btn = gr.Button('Generate', variant='primary')
            clear_btn = gr.Button('Clear prompt cache')

    with gr.Accordion('Advanced settings', open=False):
        num_step = gr.Slider(8, 32, value=DEFAULT_NUM_STEP, step=4, label='Num steps')
        guidance_scale = gr.Slider(0, 5, value=DEFAULT_GUIDANCE_SCALE, step=0.1, label='Guidance scale')
        speed = gr.Slider(0.5, 1.5, value=1.0, step=0.05, label='Speed')
        duration = gr.Number(value=None, label='Fixed duration seconds (optional)')
        denoise = gr.Checkbox(value=True, label='Denoise token')
        preprocess_prompt = gr.Checkbox(value=True, label='Preprocess reference prompt')
        postprocess_output = gr.Checkbox(value=True, label='Postprocess output')

    run_btn.click(
        generate_audio,
        inputs=[
            text, language, mode, ref_audio, ref_text, instruct,
            num_step, guidance_scale, denoise, speed, duration,
            preprocess_prompt, postprocess_output,
        ],
        outputs=[output, status],
    )
    clear_btn.click(clear_prompt_cache, outputs=[status])

print('Gradio app đã được tạo.')


In [ ]:
#@title 7) Launch Gradio public URL
demo.queue(max_size=16).launch(share=SHARE_PUBLIC_LINK, debug=True)


## Gợi ý tối ưu khi Colab bị thiếu VRAM

- Giảm `DEFAULT_NUM_STEP` xuống `12` hoặc `16`.
- Nhập sẵn `Reference text` để tránh phải tải ASR.
- Để `LOAD_ASR=False` nếu không cần tự transcription.
- Dùng reference audio khoảng 3-10 giây.
- Text dài nên chia nhỏ hoặc đặt duration hợp lý.
- Nếu gặp OOM, chạy `Runtime` → `Restart runtime`, rồi chạy lại notebook.
